In [ ]:
import logging
from ecmwf.datastores import Client
import time
import os

# Enable logging to see progress
logging.basicConfig(level="INFO")

# Create client
client = Client()
client.check_authentication()  # Verify you're logged in

years = [str(y) for y in range(2016, 2026)]  # 2016-2025
months = [f'{m:02d}' for m in range(1, 13)]
days = [f'{d:02d}' for d in range(1, 32)]

collection_id = "derived-era5-land-daily-statistics"

print(f"🚀 Submitting {len(years)} requests (non-blocking)...\n")

# Store remote objects to track requests
remotes = {}

for year in years:
    if os.path.exists(f'era5_land_daily_{year}.nc'):
        print(f"✓ era5_land_daily_{year}.nc already exists, skipping {year}.")
        continue
    request = {
        'product_type': ['reanalysis'],
        'variable': ['snow_depth_water_equivalent'],
        'year': [year],
        'month': months,
        'day': days,
        'time_zone': ['utc+00:00'],
        'daily_statistic': ['daily_mean'],
        'data_format': 'netcdf',
        'area': [68.5, 19.8, 65.3, 24.7]
    }
    
    try:
        print(f"📤 Submitting {year}...")
        remote = client.submit(collection_id, request)  # ✨ DOESN'T BLOCK!
        remotes[year] = remote
        print(f"✅ {year} submitted! Request ID: {remote.request_id}")
        time.sleep(0.5)  # Small delay between submissions
    except Exception as e:
        print(f"❌ Failed to submit {year}: {e}")

print(f"\n{'='*60}")
print(f"✅ All {len(remotes)} requests submitted to CDS queue!")
print(f"{'='*60}")
print("\n📋 Request IDs:")
for year, remote in remotes.items():
    print(f"  {year}: {remote.request_id}")

print("\n💡 To download later, use:")
print("=" * 60)
for year, remote in remotes.items():
    print(f"remote.download('era5_land_daily_{year}.nc')  # for {year}")

print("\n🌐 Monitor at: https://cds.climate.copernicus.eu/requests")

🚀 Submitting 10 requests (non-blocking)...

✓ era5_land_daily_2016.nc already exists, skipping 2016.
✓ era5_land_daily_2017.nc already exists, skipping 2017.
✓ era5_land_daily_2018.nc already exists, skipping 2018.
✓ era5_land_daily_2019.nc already exists, skipping 2019.
✓ era5_land_daily_2020.nc already exists, skipping 2020.
✓ era5_land_daily_2021.nc already exists, skipping 2021.
📤 Submitting 2022...


INFO:ecmwf.datastores.processing:Request ID is 59d8415a-09b6-4f6c-8380-a5c455172dd4


✅ 2022 submitted! Request ID: 59d8415a-09b6-4f6c-8380-a5c455172dd4
📤 Submitting 2023...


INFO:ecmwf.datastores.processing:Request ID is e064ae89-3331-4a0f-ab1f-5a2fc764cb66


✅ 2023 submitted! Request ID: e064ae89-3331-4a0f-ab1f-5a2fc764cb66
📤 Submitting 2024...


INFO:ecmwf.datastores.processing:Request ID is 953b2df3-7a50-4c5c-a875-39a33f2cfcf2


✅ 2024 submitted! Request ID: 953b2df3-7a50-4c5c-a875-39a33f2cfcf2
✓ era5_land_daily_2025.nc already exists, skipping 2025.

✅ All 3 requests submitted to CDS queue!

📋 Request IDs:
  2022: 59d8415a-09b6-4f6c-8380-a5c455172dd4
  2023: e064ae89-3331-4a0f-ab1f-5a2fc764cb66
  2024: 953b2df3-7a50-4c5c-a875-39a33f2cfcf2

💡 To download later, use:
remote.download('era5_land_daily_2022.nc')  # for 2022
remote.download('era5_land_daily_2023.nc')  # for 2023
remote.download('era5_land_daily_2024.nc')  # for 2024

🌐 Monitor at: https://cds.climate.copernicus.eu/requests


In [12]:
# query the data

import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import glob
files = sorted(glob.glob('../data/Meteorological/era5_land_daily_*.nc'))
ds = xr.open_mfdataset(files, combine='by_coords')
# write to a single file
ds.to_netcdf('../data/Meteorological/era5_land_daily_all.nc')
print(ds.data_vars)
print(ds.dims)

Data variables:
    sd       (valid_time, latitude, longitude) float32 24MB dask.array<chunksize=(366, 33, 50), meta=np.ndarray>
FrozenMappingWarningOnValuesAccess({'valid_time': 3653, 'latitude': 33, 'longitude': 50})


In [ ]:
point_15th = point.sel(valid_time=point.valid_time.dt.day == 15)

plt.figure(figsize=(14, 5))
plt.plot(point_15th.valid_time.values, point_15th.values, marker='o', markersize=3, linewidth=1)
plt.title('Snow Depth (sd) on 15th of Each Month\nLat: 67.0, Lon: 22.0')
plt.xlabel('Date')
plt.ylabel('Snow Depth (m)')
plt.grid(True)
plt.tight_layout()
plt.show()

AttributeError: 'DataArray' object has no attribute 'time'